In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.neighbors import KernelDensity
import SimpleITK as sitk

################################
# 1. Define a threshold function
################################


def compute_density(grid, points, bw, weights=None):
    """
    Compute a KDE probability density (optionally weighted).

    Args:
        grid (numpy.ndarray): 3D grid points.
        points (numpy.ndarray): 3D coordinates of points.
        bw (float): Bandwidth for KDE.
        weights (np.ndarray or None): Optional sample weights (same length as points).

    Returns:
        density (numpy.ndarray): KDE evaluated at grid.
    """
    kde = KernelDensity(kernel='gaussian', bandwidth=bw)
    kde.fit(points, sample_weight=weights)
    log_density = kde.score_samples(grid)
    return np.exp(log_density)


def generate_grid(points, bw):
    """
    Generate a 3D grid based on the range of points and half the bandwidth.
    """
    x_min, x_max = points[:, 0].min(), points[:, 0].max()
    y_min, y_max = points[:, 1].min(), points[:, 1].max()
    z_min, z_max = points[:, 2].min(), points[:, 2].max()

    # Use half the bandwidth for grid spacing
    step = bw / 2.0  
    x = np.arange(x_min, x_max, step)
    y = np.arange(y_min, y_max, step)
    z = np.arange(z_min, z_max, step)

    grid = np.array(np.meshgrid(x, y, z)).T.reshape(-1, 3)
    return grid, (len(x), len(y), len(z))


def process_odor_data(csv_folder, intensity_col, vol_th, quantile=0.1, OB_mask_path=None, use_kde_weights=False):
    """
    Process the odor data by aggregating points (without using intensities as weights).

    Args:
        csv_folder (str): Path to the folder containing odor CSV files.
        intensity_col (str): Column name for intensity filtering (used for filtering only).
        vol_th (float): Minimum volume for filtering.

    Returns:
        dict: A dictionary with odor names as keys and aggregated points as values.
    """
    csv_files = [f for f in os.listdir(csv_folder) if f.endswith('.csv')]
    odor_names = list(set(f.split('60')[0] for f in csv_files))  # Extract unique odor names
    odor_data = {}
    if OB_mask_path is None or not os.path.exists(OB_mask_path):
        raise FileNotFoundError("Olfactory bulb mask file not found. Provide a valid path.")
    OB_mask = sitk.GetArrayFromImage(sitk.ReadImage(OB_mask_path))
    OB_mask = np.squeeze(OB_mask) > 0  # Convert to boolean

    for odor_name in odor_names:
        print(f"Processing odor: {odor_name}")

        aggregated_Tpoints = []

        for csv_file in csv_files:
            if not csv_file.startswith(odor_name):
                continue

            # Read the CSV file
            selected_file_path = os.path.join(csv_folder, csv_file)
            df = pd.read_csv(selected_file_path)

            # Check for required columns
            required_columns = {'x', 'y', 'z', intensity_col, 'volume'}
            if not required_columns.issubset(df.columns):
                print(f"Skipping file '{csv_file}' due to missing required columns.")
                continue

            # Filter based on intensity and volume
            max_intensity = df[intensity_col].quantile(0.99)
            min_intensity = df[intensity_col].quantile(quantile)
            filtered_df = df[
                (df[intensity_col] >= min_intensity) &
                (df[intensity_col] <= max_intensity) &
                (df['volume'] >= vol_th)
            ]
            if filtered_df.empty:
                continue

            # Filter with mask
            filtered_df = filtered_df.copy()
            filtered_df['z'] = filtered_df['z'].round().astype(int)
            filtered_df['y'] = (filtered_df['y'] - 10).round().astype(int)  # Apply y-offset
            filtered_df['x'] = filtered_df['x'].round().astype(int)

            in_bounds = (
                (filtered_df['z'] >= 0) & (filtered_df['z'] < OB_mask.shape[0]) &
                (filtered_df['y'] >= 0) & (filtered_df['y'] < OB_mask.shape[1]) &
                (filtered_df['x'] >= 0) & (filtered_df['x'] < OB_mask.shape[2])
            )
            filtered_df = filtered_df[in_bounds]
            filtered_df['is_in_mask'] = OB_mask[filtered_df['z'], filtered_df['y'], filtered_df['x']]
            masked_spots = filtered_df[filtered_df['is_in_mask']]

            if masked_spots.empty:
                continue

            # Save CaMPARI coordinates and (optionally) their intensities
            Tpoints = masked_spots[['z', 'y', 'x']].to_numpy()
            
            if use_kde_weights:
                weights = masked_spots[intensity_col].to_numpy()
            else:
                weights = None
            
            aggregated_Tpoints.append((Tpoints, weights))

        # Concatenate all aggregated points for the current odor
        if not aggregated_Tpoints:
            continue
        # After loop
        points_all = []
        weights_all = []
        
        for pts, wts in aggregated_Tpoints:
            points_all.append(pts)
            weights_all.append(wts if wts is not None else np.ones(len(pts)))
        
        points_all = np.vstack(points_all)
        weights_all = np.concatenate(weights_all) if use_kde_weights else None
        
        odor_data[odor_name] = {'points': points_all, 'weights': weights_all}
        
    return odor_data


def p_threshold(density, p_cover=0.95):
    """
    Returns a density threshold such that the total mass in voxels above that
    threshold accounts for at least p_cover fraction of the *integral*.

    density: a 3D array or 1D array of density values.
    p_cover: fraction of mass to cover, e.g. 0.95.

    For simplicity, we do:
      1) Flatten the density
      2) Sort ascending
      3) Compute cumulative sum
      4) Find the cutoff index s.t. at least p_cover fraction is above it
    """
    # Flatten
    vals = density.ravel()
    # Sort ascending
    sort_vals = np.sort(vals)
    # Cumulative sum
    csum = np.cumsum(sort_vals)
    total = csum[-1]
    if total <= 0:
        # Edge case: if all zero or negative
        return 0
    
    csum_norm = csum / total
    
    # We want to keep the top (p_cover) portion, so find index where
    # csum_norm crosses (1 - p_cover).
    # Example: if p_cover=0.95, we want csum_norm to pass 0.05
    cutoff = 1 - p_cover
    idx = np.searchsorted(csum_norm, cutoff)
    # If idx is out of bounds, clamp it
    idx = min(max(idx, 0), len(sort_vals) - 1)
    return sort_vals[idx]

#############################################
# 2. Revised cross-correlation (Pearson) code
#############################################

def pearson_correlation_analysis(
    odor_data,
    jenkins_data,
    bw,
    area=None,
    min_correlation_threshold=0.0,
    p_cover=0.95,
    use_kde_weights=False,
    use_pearson_weight=False,
    n_perm=1000,
    n_boot=1000
):
    """
    Perform Pearson correlation analysis with permutation and bootstrap statistics.
    """
    def weighted_pearson(x, y, w):
        mean_x = np.sum(w * x) / np.sum(w)
        mean_y = np.sum(w * y) / np.sum(w)
        cov_xy = np.sum(w * (x - mean_x) * (y - mean_y)) / np.sum(w)
        var_x = np.sum(w * (x - mean_x)**2) / np.sum(w)
        var_y = np.sum(w * (y - mean_y)**2) / np.sum(w)
        return cov_xy / np.sqrt(var_x * var_y)

    # Filter Jenkins data
    if area is not None:
        jenkins_data = jenkins_data[jenkins_data['area'].isin(area)]
        if jenkins_data.empty:
            raise ValueError("No Jenkins data points match the specified area filter.")

    correlation_columns = [col for col in jenkins_data.columns if col.endswith('_correlation')]
    jenkins_data = jenkins_data[
        jenkins_data[correlation_columns].max(axis=1) >= min_correlation_threshold
    ]
    if jenkins_data.empty:
        raise ValueError("No Jenkins data points match the specified correlation threshold.")

    all_points = np.vstack(
        [data['points'] for data in odor_data.values()] +
        [np.array([eval(coord) for coord in jenkins_data['coords']])]
    )
    grid, grid_shape = generate_grid(all_points, bw)
    correlations_dict = {}

    for odor_name, data in odor_data.items():
        print(f"Computing correlations for odor: {odor_name}")
        points = data['points']
        if use_kde_weights and 'weights' not in data:
            raise ValueError(f"Missing CaMPARI weights for odor: {odor_name}")
        weights = data.get('weights') if use_kde_weights else None
        odor_density = compute_density(grid, points, bw, weights=weights).reshape(grid_shape)

        for col in correlation_columns:
            print(f"  Comparing with Jenkins correlation column: {col}")
            jenkins_column_data = jenkins_data[jenkins_data[col] >= min_correlation_threshold]
            jenkins_column_points = np.array([eval(coord) for coord in jenkins_column_data['coords']])

            if len(jenkins_column_points) == 0:
                print(f"    No valid Jenkins points for column {col}.")
                continue

            jenkins_weights = jenkins_column_data[col].to_numpy() if use_kde_weights else None
            jenkins_density = compute_density(grid, jenkins_column_points, bw, weights=jenkins_weights).reshape(grid_shape)

            flat_odor = odor_density.ravel()
            flat_jenkins = jenkins_density.ravel()

            if use_pearson_weight:
                voxel_weights = flat_odor * flat_jenkins
                R_no_mask = weighted_pearson(flat_odor, flat_jenkins, voxel_weights)
            else:
                voxel_weights = None
                R_no_mask = np.corrcoef(flat_odor, flat_jenkins)[0, 1]

            # Statistical tests for no-mask
            stats_no_mask = integrate_statistical_tests(
                R_obs=R_no_mask,
                x=flat_odor,
                y=flat_jenkins,
                w=voxel_weights,
                use_weights=use_pearson_weight,
                corr_func=weighted_pearson,
                n_perm=n_perm,
                n_boot=n_boot
            )

            th_odor = p_threshold(odor_density, p_cover)
            th_jenkins = p_threshold(jenkins_density, p_cover)
            th = min(th_odor, th_jenkins)
            mask = (odor_density > th) | (jenkins_density > th)

            if not np.any(mask):
                R_masked = np.nan
                stats_masked = None
            else:
                x_masked = odor_density[mask]
                y_masked = jenkins_density[mask]
                if use_pearson_weight:
                    mask_weights = x_masked * y_masked
                    R_masked = weighted_pearson(x_masked, y_masked, mask_weights)
                else:
                    mask_weights = None
                    R_masked = np.corrcoef(x_masked, y_masked)[0, 1]

                stats_masked = integrate_statistical_tests(
                    R_obs=R_masked,
                    x=x_masked,
                    y=y_masked,
                    w=mask_weights,
                    use_weights=use_pearson_weight,
                    corr_func=weighted_pearson,
                    n_perm=n_perm,
                    n_boot=n_boot
                )

            correlations_dict.setdefault(odor_name, {})[col] = {
                "no_mask": stats_no_mask,
                "masked": stats_masked
            }

    return correlations_dict



# Let's add permutation and bootstrap tests to the existing analysis pipeline.
# The modifications will include:
# - A permutation test: shuffle Jenkins voxel correlations and recalculate Pearson correlation to build a null distribution.
# - A bootstrap test: resample voxels with replacement and recalculate the correlation to estimate confidence intervals.

import numpy as np

def permutation_test(x, y, w=None, n_iter=1000, use_weights=False, corr_func=None):
    """
    Permutation test for Pearson correlation.

    Args:
        x, y (np.ndarray): Flattened voxel data.
        w (np.ndarray): Optional weights.
        n_iter (int): Number of permutations.
        use_weights (bool): Whether to use weighted correlation.
        corr_func (callable): Function to compute correlation.

    Returns:
        p_value (float), z_score (float), null_distribution (np.ndarray)
    """
    obs_corr = corr_func(x, y, w) if use_weights else np.corrcoef(x, y)[0, 1]
    null_dist = np.zeros(n_iter)
    for i in range(n_iter):
        y_shuffled = np.random.permutation(y)
        null_dist[i] = corr_func(x, y_shuffled, w) if use_weights else np.corrcoef(x, y_shuffled)[0, 1]

    z_score = (obs_corr - np.mean(null_dist)) / np.std(null_dist)
    p_value = np.mean(np.abs(null_dist) >= np.abs(obs_corr))
    return p_value, z_score, null_dist


def bootstrap_correlation(x, y, w=None, n_iter=1000, use_weights=False, corr_func=None):
    """
    Bootstrap test for Pearson correlation.

    Args:
        x, y (np.ndarray): Flattened voxel data.
        w (np.ndarray): Optional weights.
        n_iter (int): Number of bootstraps.
        use_weights (bool): Whether to use weighted correlation.
        corr_func (callable): Function to compute correlation.

    Returns:
        (low_CI, high_CI), bootstrapped_distributions
    """
    n = len(x)
    boot_corrs = np.zeros(n_iter)
    for i in range(n_iter):
        idx = np.random.choice(n, n, replace=True)
        x_boot = x[idx]
        y_boot = y[idx]
        w_boot = w[idx] if w is not None else None
        boot_corrs[i] = corr_func(x_boot, y_boot, w_boot) if use_weights else np.corrcoef(x_boot, y_boot)[0, 1]

    low_CI = np.percentile(boot_corrs, 2.5)
    high_CI = np.percentile(boot_corrs, 97.5)
    return (low_CI, high_CI), boot_corrs


def integrate_statistical_tests(
    R_obs,
    x,
    y,
    w,
    use_weights,
    corr_func,
    n_perm=1000,
    n_boot=1000
):
    """
    Run permutation and bootstrap tests for a given correlation result.

    Args:
        R_obs (float): Observed correlation.
        x, y (np.ndarray): Data vectors (e.g., flattened densities).
        w (np.ndarray or None): Optional weights.
        use_weights (bool): Whether weights should be used.
        corr_func (function): Function to compute weighted correlation.
        n_perm (int): Number of permutations.
        n_boot (int): Number of bootstraps.

    Returns:
        dict: {
            'R_obs': float,
            'perm_p': float,
            'perm_z': float,
            'perm_null': np.ndarray,
            'boot_CI': (float, float),
            'boot_dist': np.ndarray
        }
    """
    # Permutation test
    perm_p, perm_z, perm_null = permutation_test(
        x, y, w=w, n_iter=n_perm, use_weights=use_weights, corr_func=corr_func
    )

    # Bootstrap confidence intervals
    boot_CI, boot_dist = bootstrap_correlation(
        x, y, w=w, n_iter=n_boot, use_weights=use_weights, corr_func=corr_func
    )

    return {
        "R_obs": R_obs,
        "perm_p": perm_p,
        "perm_z": perm_z,
        "perm_null": perm_null,
        "boot_CI": boot_CI,
        "boot_dist": boot_dist,
    }

################################
# 3. Use the revised function
################################
# Paths to data
# jenkins_data = pd.read_csv(jenkins_data_path)
# vent_tele_mask_path=r'U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\imaging_data\proccessed imaging data\campari\202405_campari_analysis_CMTK\images\20241105_analysis\mapZbrain_camp_reg\ventral_telencephalon_(subpallium).tif'
# dors_tele_mask_path=r'U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\imaging_data\proccessed imaging data\campari\202405_campari_analysis_CMTK\images\20241105_analysis\mapZbrain_camp_reg\dorsal_telencephalon_(pallium).tif'

if __name__ == "__main__":
    # Paths to data
    csv_folder = r"U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\imaging_data\proccessed imaging data\campari\202405_campari_analysis_CMTK\images\20241105_analysis\mapZbrain_camp_reg\202501_transformed_csv_files"
    jenkins_data_path = r"U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\imaging_data\proccessed imaging data\campari\202405_campari_analysis_CMTK\images\20241105_analysis\mapZbrain_camp_reg\stimulus_regression_results_Tel.csv"
   # jenkins_data_path = r"U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\imaging_data\proccessed imaging data\campari\202405_campari_analysis_CMTK\images\20241105_analysis\mapZbrain_camp_reg\7dpf_corrected_valence_regression_Tel.csv"
    jenkins_data = pd.read_csv(jenkins_data_path)
    OB_mask_path=r'U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\imaging_data\proccessed imaging data\campari\202405_campari_analysis_CMTK\images\20241105_analysis\mapZbrain_camp_reg\olfactory_bulb.tif'

    # Example parameters
    bw = 5
    intensity_col = 'mean_intensity_channel_2'
    vol_th = 100
    min_quantile = 0.9
    area = ['olfactory_bulb']
    min_corr_threshold = 0.3

    # 3.1 Process odor data (same as your original or adjusted if needed)
    odor_data = process_odor_data(
        csv_folder,
        intensity_col,
        vol_th,
        min_quantile,
        OB_mask_path,
        use_kde_weights=False
    )

 
    results_bootstrapped_permutated_all_odors = pearson_correlation_analysis(
        odor_data=odor_data,
        jenkins_data=jenkins_data,
        bw=bw,
        area=area,
        min_correlation_threshold=min_corr_threshold,
        p_cover=0.9,
        use_kde_weights=False,
        use_pearson_weight=False,
        n_perm=1000,
        n_boot=1000
    )
    # # 3.3 Print or use results
    # for odor_name, correlations in results_pearson.items():
    #     print(f"\nResults for odor: {odor_name}")
    #     for col, (r_no_mask, r_masked) in correlations.items():
    #         print(f"  {col}: R_no_mask={r_no_mask:.4f}, R_masked={r_masked:.4f}")


In [ ]:
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

def visualize_pearson_correlations(results, dataset=None, output_folder=None):
    """
    Visualize Pearson correlation results (no mask vs. masked) as clustermaps.
    New input format example:
        {
            odor_name: {
                correlation_column: {
                    'no_mask': {'R_obs': float, ...},
                    'masked':  {'R_obs': float, ...}
                }
            }
        }

    Args:
        results (dict): Pearson correlation results.
        dataset (str or None): Optional tag for output file naming.
        output_folder (str or None): Folder path to save the figures. If None, show instead.
    """
    no_mask_rows = []
    masked_rows = []

    for odor_name, corr_dict in results.items():
        for corr_col, r_data in corr_dict.items():
            try:
                r_no_mask = r_data['no_mask']['R_obs']
                r_masked = r_data['masked']['R_obs']
            except KeyError:
                print(f"Skipping {odor_name} / {corr_col} due to missing keys.")
                continue

            no_mask_rows.append((odor_name, corr_col, r_no_mask))
            masked_rows.append((odor_name, corr_col, r_masked))

    # Convert to DataFrames
    df_no_mask = pd.DataFrame(no_mask_rows, columns=["Odor", "Correlation Column", "Correlation"])
    df_masked  = pd.DataFrame(masked_rows, columns=["Odor", "Correlation Column", "Correlation"])

    # Pivot the data
    matrix_no_mask = df_no_mask.pivot(index="Odor", columns="Correlation Column", values="Correlation")
    matrix_masked  = df_masked.pivot(index="Odor", columns="Correlation Column", values="Correlation")

    # Plot clustermaps
    for matrix, label, suffix in zip(
        [matrix_no_mask, matrix_masked],
        ["Pearson Correlation (No Mask)", "Pearson Correlation (Masked)"],
        ["no_mask", "masked"]
    ):
        g = sns.clustermap(
            matrix,
            cmap="viridis",
            method='average',
            metric='correlation',
            figsize=(12, 8),
            annot=False,
            fmt=".2f",
            cbar_kws={'label': f'Pearson R ({suffix.replace("_", " ").title()})'}
        )
        plt.suptitle(label, y=1.05, fontsize=16)

     #   g.ax_heatmap.set_title(label)

        if output_folder:
            filename = f"{dataset or 'data'}_pearson_corr_{suffix}.png"
            output_path = os.path.join(output_folder, filename)
            plt.savefig(output_path)
            plt.show()
        else:
            plt.show()
# Suppose 'pearson_results' is a dict with the new structure (R_no_mask, R_masked) ...


visualize_pearson_correlations(
    results=results_bootstrapped_permutated_all_odors,
    dataset="pearson masked and unmasked bw 5_all_odors",
    output_folder=r"U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\innate_behavior\figures\20241115_campari_function"
)

In [ ]:
from typing import Optional

def load_and_aggregate_preference_with_error(
    csv_path: str,
    drop_zero: bool = True,
    agg_func: str = "median"
) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    if drop_zero:
        df = df[df["Preference Score"] != 0.0]

    grouped = df.groupby("Odor")["Preference Score"]

    if agg_func == "mean":
        agg_pref = grouped.mean()
        variability = grouped.std(ddof=1) / grouped.count()**0.5  # SEM
    elif agg_func == "median":
        agg_pref = grouped.median()
        q1 = grouped.quantile(0.25)
        q3 = grouped.quantile(0.75)
        variability = (q3 - q1) / 2  # use IQR/2 as a rough symmetric error
    else:
        raise ValueError("agg_func must be 'mean' or 'median'")

    df_pref = pd.DataFrame({
        "Odor": agg_pref.index,
        "Mean_Pref": agg_pref.values,
        "SEM_Pref": variability.values,  # This is SEM or IQR/2
        "N": grouped.count().values
    })

    return df_pref


def correlate_correlations_with_preference(
    results: dict,
    preference_csv: str,
    *,
    use_masked: bool = False,
    pos_pattern: str = r"\bpos",
    neg_pattern: str = r"\bneg",
    plot: bool = True,
    output_folder: Optional[str] = None,
    pref_agg: str = "median",          # NEW: "mean" or "median"
    drop_zero_pref: bool = False,     # NEW: remove Preference==0.0 rows?
):
    """As before, but now handles per-file Preference CSV with zero-filtering."""
    # ------------------------------------------------------------------
    # 0) load & collapse behavioural CSV
    # ------------------------------------------------------------------
    df_pref = load_and_aggregate_preference_with_error(
        preference_csv, drop_zero=drop_zero_pref, agg_func=pref_agg 
    )


    # diagnostic printout
    if drop_zero_pref:
        n_removed_total = int(removed.sum())
        print(f"\nRemoved {n_removed_total} rows with Preference Score == 0.0")
        if not removed.empty:
            print("Break-down by odor:")
            display(removed)        # pretty table in JupyterLab

    # ------------------------------------------------------------------
    # 1) aggregate imaging correlations
    # ------------------------------------------------------------------
    rows = []
    for odor, corr_dict in results.items():
        pos_vals, neg_vals = [], []
        for col, (r_no_mask, r_masked) in corr_dict.items():
            r_val = r_masked if use_masked else r_no_mask
            if re.search(pos_pattern, col, flags=re.I):
                pos_vals.append(r_val)
            elif re.search(neg_pattern, col, flags=re.I):
                neg_vals.append(r_val)

        rows.append(
            (
                odor,
                np.nanmean(pos_vals) if pos_vals else np.nan,
                np.nanmean(neg_vals) if neg_vals else np.nan,
            )
        )

    df_corr = pd.DataFrame(rows, columns=["Odor", "Mean_Positive_R", "Mean_Negative_R"])

    # ------------------------------------------------------------------
    # 2) merge & analyse (unchanged)
    # ------------------------------------------------------------------
    df = df_corr.merge(df_pref, on="Odor", how="inner")
    df = df.rename(columns={"Mean_Pref": "Preference Score"})

    stats = {}
    for col, label in (("Mean_Positive_R", "positive"),
                       ("Mean_Negative_R", "negative")):
        sub = df[[col, "Preference Score"]].dropna()
        r, p = pearsonr(sub[col], sub["Preference Score"]) if len(sub) > 1 else (np.nan, np.nan)
        stats[label] = (r, p)

        if plot and len(sub) > 1:
            sns.set(style="whitegrid")
            ax = sns.regplot(data=sub, x=col, y="Preference Score")
            ax.set_title(f"{label.capitalize()} R vs Preference   (r = {r:.2f}, p = {p:.3g})")
            ax.set_xlabel(f"{col} ({'masked' if use_masked else 'no-mask'})")
            ax.set_ylabel("Preference Score")
            plt.tight_layout()
            if output_folder:
                os.makedirs(output_folder, exist_ok=True)
                fname = os.path.join(output_folder, f"{label}_r_vs_preference.png")
                plt.savefig(fname, dpi=300)
            plt.show()

    return df, stats

def correlate_correlations_with_preference_with_CI(
    results: dict,
    preference_csv: str,
    *,
    use_masked: bool = True,
    pos_pattern: str = r"\bpos",
    neg_pattern: str = r"\bneg",
    plot: bool = True,
    output_folder: Optional[str] = None,
    agg_func: str = "median",
    drop_zero_pref: bool = False,
):
    """
    Extended version: includes bootstrap CI and permutation p-values.
    """
    df_pref = load_and_aggregate_preference_with_error(
        preference_csv, drop_zero=drop_zero_pref,agg_func=pref_agg 
    )

    if drop_zero_pref and not removed.empty:
        print(f"\nRemoved {int(removed.sum())} rows with Preference Score == 0.0")
        print("Break-down by odor:")
        display(removed)

    rows = []
    for odor, corr_dict in results.items():
        pos_vals, neg_vals = [], []
        pos_ci, neg_ci = [], []
        pos_pval, neg_pval = [], []

        for col, stat_dict in corr_dict.items():
            stat = stat_dict["masked"] if use_masked else stat_dict["no_mask"]
            r_val = stat["R_obs"]
            ci = stat["boot_CI"]
            pval = stat["perm_p"]

            if re.search(pos_pattern, col, flags=re.I):
                pos_vals.append(r_val)
                pos_ci.append(ci)
                pos_pval.append(pval)
            elif re.search(neg_pattern, col, flags=re.I):
                neg_vals.append(r_val)
                neg_ci.append(ci)
                neg_pval.append(pval)

        rows.append((
            odor,
            np.nanmean(pos_vals) if pos_vals else np.nan,
            np.nanmean(neg_vals) if neg_vals else np.nan,
            np.nanmean([ci[0] for ci in pos_ci]) if pos_ci else np.nan,
            np.nanmean([ci[1] for ci in pos_ci]) if pos_ci else np.nan,
            np.nanmean([ci[0] for ci in neg_ci]) if neg_ci else np.nan,
            np.nanmean([ci[1] for ci in neg_ci]) if neg_ci else np.nan,
            np.nanmean(pos_pval) if pos_pval else np.nan,
            np.nanmean(neg_pval) if neg_pval else np.nan,
        ))

    df_corr = pd.DataFrame(rows, columns=[
        "Odor", "Mean_Positive_R", "Mean_Negative_R",
        "Pos_CI_Low", "Pos_CI_High", "Neg_CI_Low", "Neg_CI_High",
        "Pos_Perm_P", "Neg_Perm_P"
    ])

    df = df_corr.merge(df_pref, on="Odor", how="inner")
    df = df.rename(columns={"Mean_Pref": "Preference Score"})

    # Compute ΔR and ΔCI
    df["Delta_R"] = df["Mean_Positive_R"] - df["Mean_Negative_R"]
    df["Delta_CI_Low"] = df["Pos_CI_Low"] - df["Neg_CI_High"]
    df["Delta_CI_High"] = df["Pos_CI_High"] - df["Neg_CI_Low"]

    stats = {}
    for col, label in (("Mean_Positive_R", "positive"), ("Mean_Negative_R", "negative")):
        sub = df[[col, "Preference Score"]].dropna()
        r, p = pearsonr(sub[col], sub["Preference Score"]) if len(sub) > 1 else (np.nan, np.nan)
        stats[label] = (r, p)

        if plot and len(sub) > 1:
            plt.figure()
            plt.errorbar(
                sub[col], sub["Preference Score"],
                xerr=[
                    sub[col] - df[f"{'Pos' if label == 'positive' else 'Neg'}_CI_Low"],
                    df[f"{'Pos' if label == 'positive' else 'Neg'}_CI_High"] - sub[col]
                ],
                fmt='o', capsize=3, alpha=0.6, ecolor='gray'
            )
            sns.regplot(data=sub, x=col, y="Preference Score", scatter=False, color='black')
            plt.title(f"{label.capitalize()} R vs Preference (r = {r:.2f}, p = {p:.3g})")
            plt.xlabel(f"{col} ({'masked' if use_masked else 'no-mask'})")
            plt.ylabel("Preference Score")
            plt.tight_layout()
            if output_folder:
                os.makedirs(output_folder, exist_ok=True)
                fname = os.path.join(output_folder, f"{label}_r_vs_preference_CI.png")
                plt.savefig(fname, dpi=300)
            plt.show()

    # ΔR vs preference
    sub = df[["Delta_R", "Preference Score", "Delta_CI_Low", "Delta_CI_High"]].dropna()
    r_p, p_p = pearsonr(sub["Delta_R"], sub["Preference Score"])
    r_s, p_s = spearmanr(sub["Delta_R"], sub["Preference Score"], nan_policy="omit")
    stats["delta"] = {"pearson": (r_p, p_p), "spearman": (r_s, p_s)}

    if plot and len(sub) > 1:
        plt.figure()
        plt.errorbar(
            sub["Delta_R"], sub["Preference Score"],
            xerr=[
                sub["Delta_R"] - sub["Delta_CI_Low"],
                sub["Delta_CI_High"] - sub["Delta_R"]
            ],
            fmt='o', capsize=3, alpha=0.6, ecolor='gray'
        )
        sns.regplot(data=sub, x="Delta_R", y="Preference Score", scatter=False, color='black')
        plt.title(
            f"ΔR vs Preference\n"
            f"Pearson r = {r_p:.2f}, p = {p_p:.3g} | Spearman ρ = {r_s:.2f}, p = {p_s:.3g}"
        )
        plt.xlabel("ΔR (Mean_Positive_R − Mean_Negative_R)")
        plt.ylabel("Preference Score")
        plt.tight_layout()
        if output_folder:
            fname = os.path.join(output_folder, "delta_r_vs_preference_CI.png")
            plt.savefig(fname, dpi=300)
        plt.show()

    return df, stats


In [ ]:
# Imaging-results dict produced earlier in the notebook:
# results_pearson_updated_regressors  ← must already exist
import os, re
from scipy.stats import pearsonr, spearmanr

preference_csv = (
    r"U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\innate_behavior\figures"
    r"\20241115_behavior\2020505_plots\processed_data_and_plots\median_values_inOdor.csv"
)
preference_csv_detailed = (
    r"U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\innate_behavior\figures"
    r"\20241115_behavior\2020505_plots\processed_data_and_plots\prefernce_scores_detailed.csv"
)
# Optional folder to collect the two PNGs; leave as '' or None to skip saving
fig_output = (
    r"U:\Scientific Data\RG-AS04-Data01\Oded_Mayseless\innate_behavior\figures"
    r"\20241115_campari_function"
)
# df_summary, r_stats = correlate_correlations_with_preference(
#     results=results_pearson_updated_regressors_weighted,
#     preference_csv=preference_csv,
#     use_masked=True,      # set to True if you need masked R values
#     plot=False,             # plots appear inline; turn off if running repeatedly
#     output_folder=fig_output,
# )
pref_agg = 'mean'
df_summary, r_stats = correlate_correlations_with_preference_with_CI(
    results=results_bootstrapped_permutated_all_odors,
    preference_csv=preference_csv_detailed,
    use_masked=True,
    plot=False,
    output_folder=fig_output,
    drop_zero_pref= False,
    agg_func=pref_agg 
)
#display(df_summary)        # nice table in JupyterLab
print("\nPearson r (p-value)  → ", r_stats)


In [ ]:
def plot_forest_with_CI(df, value_col, low_col, high_col, title, color, output_path=None):
    """
    Generate a forest plot with 95% confidence intervals.
    """
    df_sorted = df.sort_values(value_col)
    y_pos = np.arange(len(df_sorted))

    plt.figure(figsize=(8, len(df_sorted) * 0.4))
    plt.errorbar(
        df_sorted[value_col], y_pos,
        xerr=[df_sorted[value_col] - df_sorted[low_col], df_sorted[high_col] - df_sorted[value_col]],
        fmt='o', color=color, ecolor='gray', capsize=4
    )
    plt.yticks(y_pos, df_sorted['Odor'])
    plt.axvline(0, linestyle='--', color='black', linewidth=0.7)
    plt.xlabel("Correlation (R)")
    plt.title(title)
    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=300)
    plt.show()


def plot_delta_R_vs_preference(
    df,
    output_path=None,
    show_permutation=True,
    show_bootstrap_CI=True
):
    """
    Plot ΔR vs Preference Score with optional bootstrap CI and permutation significance indicators.
    Handles missing SEM_Pref column gracefully.
    """
    df = df.rename(columns={"Mean_Pref": "Preference Score"})

    # Warn if SEM_Pref is missing
    sem_available = "SEM_Pref" in df.columns
    if not sem_available:
        print("⚠️ Warning: 'SEM_Pref' column not found — plotting without preference error bars.")
    
    # Define core and optional columns separately
    core_cols = ["Delta_R", "Preference Score", "Delta_CI_Low", "Delta_CI_High"]
    optional_cols = ["SEM_Pref"] if "SEM_Pref" in df.columns else []
    
    # Drop rows only if core columns are NaN
    sub = df[core_cols + optional_cols].dropna(subset=core_cols)

    # Compute correlation stats
    r_p, p_p = pearsonr(sub["Delta_R"], sub["Preference Score"])
    r_s, p_s = spearmanr(sub["Delta_R"], sub["Preference Score"], nan_policy="omit")

    # Plot
    plt.figure(figsize=(6, 5))
    if show_bootstrap_CI:
        yerr = sub["SEM_Pref"] if sem_available else None
        plt.errorbar(
            sub["Delta_R"], sub["Preference Score"],
            xerr=[
                sub["Delta_R"] - sub["Delta_CI_Low"],
                sub["Delta_CI_High"] - sub["Delta_R"]
            ],
            yerr=yerr,
            fmt='o', capsize=4, alpha=0.7, ecolor='gray'
        )
    else:
        plt.plot(sub["Delta_R"], sub["Preference Score"], 'o', alpha=0.7)

    sns.regplot(data=sub, x="Delta_R", y="Preference Score", scatter=False, color='black')
    plt.title(f"ΔR vs Preference\n Spearman ρ = {r_s:.2f}, p = {p_s:.3g}")
    plt.xlabel("ΔR (Mean_Positive_R − Mean_Negative_R)")
    plt.ylabel("Preference Score")

    # Annotate permutation p-values if available
    if show_permutation and "Pos_Perm_P" in df.columns and "Neg_Perm_P" in df.columns:
        for _, row in sub.iterrows():
            pos_p = row.get("Pos_Perm_P", np.nan)
            neg_p = row.get("Neg_Perm_P", np.nan)
            if pd.notna(pos_p) and pd.notna(neg_p):
                pval_str = f"(p⁺={pos_p:.2g}, p⁻={neg_p:.2g})"
                plt.annotate(
                    pval_str,
                    xy=(row["Delta_R"], row["Preference Score"]),
                    xytext=(4, -4), textcoords='offset points',
                    fontsize=7, color='gray'
                )

    plt.tight_layout()
    if output_path:
        plt.savefig(output_path, dpi=300)
    plt.show()


In [ ]:
# plot_forest_with_CI(
#     df=df_summary,
#     value_col="Mean_Positive_R",
#     low_col="Pos_CI_Low",
#     high_col="Pos_CI_High",
#     title="Positive R values with CI",
#     color="blue",
#     output_path=os.path.join(fig_output, "forest_positive_R_CI.png")
# )
plot_delta_R_vs_preference(
    df=df_summary,
    output_path=os.path.join(fig_output, "delta_r_vs_preference_fullstats.png"),
    show_permutation=True,
    show_bootstrap_CI=True
)
print(f'saved to{fig_output}')